# 05 — Modelagem Estatística
**O ponto alto do projeto:** Análise de Sobrevivência com Kaplan-Meier.

Análises:
- Curva de Kaplan-Meier estratificada por grupo de dose
- Teste de Log-Rank entre grupos
- Regressão de Cox (modelo multivariado)
- Regressão Logística: P(hospitalização) ~ dose

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/processed/vitimas_clean.csv")
dosimetria = pd.read_csv("../data/processed/dosimetria_clean.csv")

print(f"Dataset: {len(df)} pacientes | {df['obito'].sum()} obitos | {df['internado'].sum()} internacoes")


## 5.1 — Curva de Kaplan-Meier (Ponto Principal do Projeto)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Analise de Sobrevivencia — Acidente Radiologico de Goiania (1987)",
             fontsize=13, fontweight='bold')

grupos_km = {
    'alto':     {'color': '#D32F2F', 'label': 'Alta exposicao (>1000 mSv, n=13)'},
    'moderado': {'color': '#F57C00', 'label': 'Exposicao moderada (100-1000 mSv, n=36)'},
    'leve':     {'color': '#388E3C', 'label': 'Baixa exposicao (<100 mSv, n=200)'}
}

kmf = KaplanMeierFitter()

# Kaplan-Meier por grupo
for grupo, cfg in grupos_km.items():
    mask = df['grupo_exposicao'] == grupo
    kmf.fit(
        df[mask]['dias_desfecho'],
        event_observed=df[mask]['obito'],
        label=cfg['label']
    )
    kmf.plot_survival_function(ax=axes[0], color=cfg['color'], linewidth=2.5, ci_show=True)

axes[0].set_xlabel("Dias apos exposicao")
axes[0].set_ylabel("Probabilidade de sobrevivencia")
axes[0].set_title("Curva de Kaplan-Meier por Grupo de Dose")
axes[0].set_ylim(0, 1.05)
axes[0].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Mediana')
axes[0].legend(loc='lower left', fontsize=8)

# KM por categoria de dose (4 grupos)
df['cat_dose4'] = pd.cut(
    df['dose_mSv'],
    bins=[0, 50, 200, 1000, 10001],
    labels=['< 50 mSv', '50-200 mSv', '200-1000 mSv', '> 1000 mSv']
)
cores4 = ['#388E3C', '#FBC02D', '#F57C00', '#D32F2F']
for cat, cor in zip(['< 50 mSv', '50-200 mSv', '200-1000 mSv', '> 1000 mSv'], cores4):
    mask = df['cat_dose4'] == cat
    if mask.sum() > 0:
        kmf.fit(df[mask]['dias_desfecho'], event_observed=df[mask]['obito'], label=cat)
        kmf.plot_survival_function(ax=axes[1], color=cor, linewidth=2.5, ci_show=False)

axes[1].set_xlabel("Dias apos exposicao")
axes[1].set_ylabel("Probabilidade de sobrevivencia")
axes[1].set_title("Kaplan-Meier por Faixa de Dose (4 grupos)")
axes[1].set_ylim(0, 1.05)
axes[1].legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.savefig("../reports/figures/05_kaplan_meier.png", dpi=150, bbox_inches='tight')
plt.show()
print("Curva de Kaplan-Meier salva")


## 5.2 — Teste de Log-Rank

In [ ]:
# Comparar grupos par a par
alto  = df[df['grupo_exposicao'] == 'alto']
mod   = df[df['grupo_exposicao'] == 'moderado']
leve  = df[df['grupo_exposicao'] == 'leve']

print("=== TESTE DE LOG-RANK ===\n")

resultado_alto_leve = logrank_test(
    alto['dias_desfecho'], leve['dias_desfecho'],
    event_observed_A=alto['obito'], event_observed_B=leve['obito']
)
print(f"Alto vs. Leve:     p = {resultado_alto_leve.p_value:.6f}  {'*** SIGNIFICATIVO' if resultado_alto_leve.p_value < 0.05 else 'nao significativo'}")

resultado_alto_mod = logrank_test(
    alto['dias_desfecho'], mod['dias_desfecho'],
    event_observed_A=alto['obito'], event_observed_B=mod['obito']
)
print(f"Alto vs. Moderado: p = {resultado_alto_mod.p_value:.6f}  {'*** SIGNIFICATIVO' if resultado_alto_mod.p_value < 0.05 else 'nao significativo'}")

resultado_mod_leve = logrank_test(
    mod['dias_desfecho'], leve['dias_desfecho'],
    event_observed_A=mod['obito'], event_observed_B=leve['obito']
)
print(f"Moderado vs. Leve: p = {resultado_mod_leve.p_value:.6f}  {'*** SIGNIFICATIVO' if resultado_mod_leve.p_value < 0.05 else 'nao significativo'}")

# Teste multivariado
from lifelines.statistics import multivariate_logrank_test
multi = multivariate_logrank_test(df['dias_desfecho'], df['grupo_exposicao'], df['obito'])
print(f"\nLog-Rank Multivariado (3 grupos): p = {multi.p_value:.6f}")
print(f"Interpretacao: {'As curvas de sobrevivencia diferem SIGNIFICATIVAMENTE entre os grupos.' if multi.p_value < 0.05 else 'Sem diferenca significativa.'}")


## 5.3 — Modelo de Cox (Regressão Multivariada)

In [ ]:
# Preparar dados para Cox
df_cox = df[['dias_desfecho', 'obito', 'dose_mSv', 'idade']].copy()
df_cox['sexo_M'] = (df['sexo'] == 'M').astype(int)
df_cox['log_dose'] = np.log1p(df_cox['dose_mSv'])
df_cox = df_cox.dropna()

cph = CoxPHFitter()
cph.fit(df_cox[['dias_desfecho', 'obito', 'log_dose', 'idade', 'sexo_M']],
        duration_col='dias_desfecho', event_col='obito')

print("=== MODELO DE COX — RESULTADOS ===\n")
cph.print_summary()

fig, ax = plt.subplots(figsize=(8, 4))
cph.plot(ax=ax)
ax.set_title("Hazard Ratios — Modelo de Cox")
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("../reports/figures/05_cox_hazard_ratios.png", dpi=150, bbox_inches='tight')
plt.show()


## 5.4 — Regressão Logística: P(hospitalização) ~ dose

In [ ]:
X = df[['dose_mSv', 'idade']].copy()
X['sexo_M'] = (df['sexo'] == 'M').astype(int)
X['log_dose'] = np.log1p(X['dose_mSv'])
X = X[['log_dose', 'idade', 'sexo_M']]
y = df['internado']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(max_iter=500)
lr.fit(X_train_sc, y_train)
y_pred = lr.predict(X_test_sc)
y_prob = lr.predict_proba(X_test_sc)[:, 1]
auc = roc_auc_score(y_test, y_prob)

print("=== REGRESSAO LOGISTICA ===")
print(classification_report(y_test, y_pred, target_names=['Nao internado', 'Internado']))
print(f"AUC-ROC: {auc:.4f}")

fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#1565C0', linewidth=2.5, label=f'AUC = {auc:.3f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
ax.set_xlabel("Taxa de Falso Positivo")
ax.set_ylabel("Taxa de Verdadeiro Positivo")
ax.set_title("Curva ROC — Predicao de Internacao")
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig("../reports/figures/05_roc_curve.png", dpi=150, bbox_inches='tight')
plt.show()
